**Aharouni Refael, Aye William (DIA1)** <br><br>
<h1 style='text-align:center; font-weight:bold;'>TD4 — Knowledge Base Construction & Expansion</h1><br>


## Lab 4 — KB Construction & SPARQL Expansion

### Objectives
In this lab we transform our raw NER output into a proper **Knowledge Graph** by:
1. Linking extracted entities to **Wikidata QIDs** (globally unique identifiers)
2. Expanding the graph using **SPARQL queries** against the Wikidata endpoint
3. Creating a **predicate alignment** to standard vocabularies

### Why Link to Wikidata?
The NER output from Lab 1 contains raw text strings like 'James Clerk Maxwell' and 'University of Cambridge'.
But text strings are **ambiguous and fragile**:
- "Maxwell" could mean the person, the equations, or the unit of magnetic flux
- "Cambridge" could be the city in England or Massachusetts, or the university
- Spelling variations ('Ampere' vs 'Ampère') break exact matching

By linking each entity to a **Wikidata QID** (e.g., `wd:Q9095` for Maxwell), we get:
- **Unique global identifiers** — no ambiguity, no duplicates
- **Structured properties** — occupation, birth place, awards are already curated
- **Cross-language links** — the same QID works in all languages
- **Rich 2-hop neighborhood** — we can discover related entities not in our original crawl

### Expansion Strategy
Starting from ~200 seed entities (from NER), we expand in 3 waves:
1. **1-hop**: For each seed entity, fetch all 15 key properties (occupation, birth place, awards, etc.)
2. **2-hop**: For newly discovered entities, fetch their properties too
3. **Occupation expansion**: Find more scientists/physicists connected to our seed entities

This grows our graph from ~200 entities to **~52,000 triples**.

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
from collections import Counter
from rdflib import Graph, URIRef
import statistics

SPARQL_URL = "https://query.wikidata.org/sparql"

headers = {
    "User-Agent": "KBExpansionBot/1.0 william.aye75@gmail.com)"
}

# -----------------------------
# ALLOWED RELATIONS (CRITICAL)
# Extended with more predicates to enrich the graph
# -----------------------------
ALLOWED_PREDICATES = {
    "http://www.wikidata.org/prop/direct/P31",   # instance of
    "http://www.wikidata.org/prop/direct/P279",  # subclass of
    "http://www.wikidata.org/prop/direct/P361",  # part of
    "http://www.wikidata.org/prop/direct/P527",  # has part
    "http://www.wikidata.org/prop/direct/P166",  # award received
    "http://www.wikidata.org/prop/direct/P106",  # occupation
    "http://www.wikidata.org/prop/direct/P19",   # place of birth
    "http://www.wikidata.org/prop/direct/P69",   # educated at
    "http://www.wikidata.org/prop/direct/P17",   # country
    "http://www.wikidata.org/prop/direct/P131",  # located in
    "http://www.wikidata.org/prop/direct/P27",   # country of citizenship
    "http://www.wikidata.org/prop/direct/P21",   # sex or gender
    "http://www.wikidata.org/prop/direct/P800",  # notable work
    "http://www.wikidata.org/prop/direct/P108",  # employer
    "http://www.wikidata.org/prop/direct/P1412", # languages spoken
}

# -----------------------------
# SPARQL EXECUTION
# -----------------------------
def run_query(query):
    response = requests.get(
        SPARQL_URL,
        params={"query": query, "format": "json"},
        headers=headers,
        timeout=60
    )
    response.raise_for_status()
    return response.json()


# -----------------------------
# SAFE EXTRACTION
# -----------------------------
def extract_triples(results):
    triples = set()
    for r in results["results"]["bindings"]:
        try:
            s = r["s"]["value"]
            p = r["p"]["value"]
            o = r["o"]["value"]
            triples.add((s, p, o))
        except:
            continue
    return triples


# -----------------------------
# BULK 1-HOP (ANCHOR DOMAIN)
# -----------------------------
def bulk_1hop(limit=50000):
    query = """
    SELECT ?s ?p ?o WHERE {
      ?s wdt:P31 wd:Q5 .
      ?s ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
      FILTER(STRSTARTS(STR(?o), "http://www.wikidata.org/entity/"))
    }
    LIMIT """ + str(limit)
    return extract_triples(run_query(query))


# -----------------------------
# PREDICATE EXPANSION
# -----------------------------
def bulk_predicates(limit=30000):
    query = """
    SELECT ?s ?p ?o WHERE {
      ?s wdt:P106 ?o .
      BIND(wdt:P106 AS ?p)
      FILTER(STRSTARTS(STR(?o), "http://www.wikidata.org/entity/"))
    }
    LIMIT """ + str(limit)
    return extract_triples(run_query(query))


# -----------------------------
# CONTROLLED 2-HOP
# -----------------------------
def bulk_2hop(limit=30000):
    query = """
    SELECT ?s ?p ?o WHERE {
      ?person wdt:P166 ?award .
      ?award ?p ?o .
      BIND(?award AS ?s)
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
      FILTER(STRSTARTS(STR(?o), "http://www.wikidata.org/entity/"))
    }
    LIMIT """ + str(limit)
    return extract_triples(run_query(query))


# -----------------------------
# OCCUPATION EXPANSION (NEW)
# Fetch properties of occupation entities
# to densify the graph
# -----------------------------
def bulk_occupation_expand(limit=50000):
    query = """
    SELECT ?s ?p ?o WHERE {
      ?person wdt:P106 ?s .
      ?s ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
      FILTER(STRSTARTS(STR(?o), "http://www.wikidata.org/entity/"))
    }
    LIMIT """ + str(limit)
    return extract_triples(run_query(query))


# -----------------------------
# MAIN PIPELINE WITH tqdm
# -----------------------------
all_triples = set()

print("Running bulk 1-hop...")
triples_1 = bulk_1hop(limit=200000)
all_triples.update(triples_1)
print(f"  -> {len(triples_1)} triples")

print("Running predicate expansion...")
triples_2 = bulk_predicates(limit=200000)
all_triples.update(triples_2)
print(f"  -> {len(triples_2)} triples")

print("Running 2-hop expansion...")
triples_3 = bulk_2hop(limit=200000)
all_triples.update(triples_3)
print(f"  -> {len(triples_3)} triples")

print("Running occupation expansion...")
triples_4 = bulk_occupation_expand(limit=50000)
all_triples.update(triples_4)
print(f"  -> {len(triples_4)} triples")

print(f"\nTotal before filtering: {len(all_triples)} triples")

# -----------------------------
# CLEANING + FILTERING (CRITICAL)
# -----------------------------
print("\nCleaning triples...")

cleaned_triples = set()

for (s, p, o) in tqdm(all_triples, desc="Filtering triples"):
    if (
        s != o
        and o.startswith("http://www.wikidata.org/entity/Q")
        and s.startswith("http://www.wikidata.org/entity/Q")
        and p in ALLOWED_PREDICATES
    ):
        cleaned_triples.add((s, p, o))

all_triples = cleaned_triples
print(f"After cleaning: {len(all_triples)} triples")


# -----------------------------
# ITERATIVE ENTITY DEGREE FILTERING
# Keep only entities with sufficient connections
# so that KGE models can learn meaningful embeddings.
# Entities with degree < MIN_DEGREE are removed
# iteratively until the graph stabilizes.
# -----------------------------
MIN_DEGREE = 5

print(f"\nApplying iterative degree filtering (min_degree={MIN_DEGREE})...")

for iteration in range(10):
    degree = Counter()
    for s, p, o in all_triples:
        degree[s] += 1
        degree[o] += 1

    valid_entities = {e for e, d in degree.items() if d >= MIN_DEGREE}
    prev_size = len(all_triples)

    all_triples = {
        (s, p, o) for s, p, o in all_triples
        if s in valid_entities and o in valid_entities
    }

    print(f"  Iter {iteration+1}: {len(all_triples)} triples, {len(valid_entities)} entities")

    if len(all_triples) == prev_size:
        print("  Stable. Stopping.")
        break


# -----------------------------
# FINAL STATS
# -----------------------------
entities = set()
relations = set()

for s, p, o in all_triples:
    entities.add(s)
    entities.add(o)
    relations.add(p)

print("\nFINAL STATS")
print("Triples:", len(all_triples))
print("Entities:", len(entities))
print("Relations:", len(relations))

degree = Counter()
for s, p, o in all_triples:
    degree[s] += 1
    degree[o] += 1

degrees = list(degree.values())
print(f"Avg degree: {sum(degrees)/len(degrees):.1f}")
print(f"Median degree: {statistics.median(degrees):.1f}")


# -----------------------------
# EXPORT CSV
# -----------------------------
df = pd.DataFrame(list(all_triples), columns=["subject", "predicate", "object"])
df.to_csv("../data/expanded_kb.csv", index=False)
print("\nSaved as expanded_kb.csv")


# -----------------------------
# EXPORT RDF (used directly by TD5)
# -----------------------------
g = Graph()
for s, p, o in tqdm(all_triples, desc="Building RDF graph"):
    g.add((URIRef(s), URIRef(p), URIRef(o)))

g.serialize(destination="../kg_artifacts/expanded_kb.rdf", format="turtle")
g.serialize(destination="../kg_artifacts/expanded_kb.nt", format="nt")

print("Saved as expanded_kb.rdf and expanded_kb.nt")


## Entity & Predicate Alignment

### Why Predicate Alignment?
Wikidata uses its own property identifiers (`wdt:P106`, `wdt:P19`, etc.) which are
machine-readable but **not self-documenting**. Nobody knows what `P106` means without looking it up.

We create a **predicate alignment** that maps each Wikidata property to a well-known vocabulary:

| Wikidata | Label | Aligned To | Why |
|----------|-------|-----------|-----|
| wdt:P31 | instance of | rdf:type | Standard RDF class membership |
| wdt:P279 | subclass of | rdfs:subClassOf | Standard RDF class hierarchy |
| wdt:P106 | occupation | schema:hasOccupation | Schema.org is the most-used web vocabulary |
| wdt:P19 | place of birth | schema:birthPlace | Same — widely adopted by search engines |
| wdt:P69 | educated at | schema:alumniOf | Describes person-institution relationships |
| wdt:P166 | award received | schema:award | Links people to their achievements |

This alignment makes our KB **interoperable** with other Linked Open Data on the web.
It is stored in `kg_artifacts/alignment.ttl` using `owl:equivalentProperty` and `skos:exactMatch`.

In [ ]:
# Predicate alignment: map Wikidata IDs to human-readable labels
# This is used in TD6 for the RAG schema summary

PREDICATE_ALIGNMENT = {
    'http://www.wikidata.org/prop/direct/P31':   {'label': 'instance of',         'td1_verbs': ['is a', 'is an']},
    'http://www.wikidata.org/prop/direct/P279':  {'label': 'subclass of',          'td1_verbs': ['is a type of']},
    'http://www.wikidata.org/prop/direct/P106':  {'label': 'occupation',           'td1_verbs': ['works as', 'is a']},
    'http://www.wikidata.org/prop/direct/P19':   {'label': 'place of birth',       'td1_verbs': ['was born in', 'born in']},
    'http://www.wikidata.org/prop/direct/P69':   {'label': 'educated at',          'td1_verbs': ['studied at', 'attended']},
    'http://www.wikidata.org/prop/direct/P166':  {'label': 'award received',       'td1_verbs': ['received', 'won', 'awarded']},
    'http://www.wikidata.org/prop/direct/P27':   {'label': 'country of citizenship','td1_verbs': ['is from', 'is a citizen of']},
    'http://www.wikidata.org/prop/direct/P108':  {'label': 'employer',             'td1_verbs': ['works for', 'employed by']},
    'http://www.wikidata.org/prop/direct/P17':   {'label': 'country',              'td1_verbs': ['located in', 'in']},
    'http://www.wikidata.org/prop/direct/P131':  {'label': 'located in',           'td1_verbs': ['situated in', 'part of']},
    'http://www.wikidata.org/prop/direct/P21':   {'label': 'sex or gender',        'td1_verbs': []},
    'http://www.wikidata.org/prop/direct/P800':  {'label': 'notable work',         'td1_verbs': ['wrote', 'discovered', 'invented']},
    'http://www.wikidata.org/prop/direct/P1412': {'label': 'languages spoken',     'td1_verbs': ['speaks', 'wrote in']},
}

import pandas as pd
df_alignment = pd.DataFrame([
    {'Wikidata URI': k, 'Label': v['label'], 'TD1 Verbs': ', '.join(v['td1_verbs']) or '(structural)'}
    for k, v in PREDICATE_ALIGNMENT.items()
])
print('Predicate Alignment Table:')
print(df_alignment.to_string(index=False))


## Final KB Statistics

After expansion and filtering, we compute key statistics to validate our KB:
- **Triple count**: How many facts are in the graph?
- **Entity count**: How many unique entities (Wikidata QIDs)?
- **Predicate count**: How many distinct relationship types?
- **Connectivity**: Is the graph a single connected component or fragmented?

These statistics help us assess the graph's quality before using it for
reasoning (Lab 5) and RAG (Lab 6). A sparse, disconnected graph would produce
poor embeddings and unreliable SPARQL answers.

In [ ]:
# Display final KB statistics from the exported RDF file
from rdflib import Graph
from collections import Counter
import pandas as pd

g_check = Graph()
g_check.parse('../kg_artifacts/expanded_kb.rdf', format='turtle')

# Count predicates
pred_counts = Counter(str(p) for s, p, o in g_check)

# Count entities
entities = set(str(s) for s, p, o in g_check) | set(str(o) for s, p, o in g_check)
wikidata_entities = {e for e in entities if '/entity/Q' in e}

print('=== Final KB Statistics ===')
print(f'Total triples:       {len(g_check)}')
print(f'Distinct entities:   {len(wikidata_entities)}')
print(f'Distinct predicates: {len(pred_counts)}')
print()
print('Top 10 predicates by frequency:')
from rdflib import URIRef
PRED_LABELS = {
    'http://www.wikidata.org/prop/direct/P106': 'occupation',
    'http://www.wikidata.org/prop/direct/P166': 'award received',
    'http://www.wikidata.org/prop/direct/P1412': 'languages spoken',
    'http://www.wikidata.org/prop/direct/P27': 'country of citizenship',
    'http://www.wikidata.org/prop/direct/P31': 'instance of',
    'http://www.wikidata.org/prop/direct/P21': 'sex or gender',
    'http://www.wikidata.org/prop/direct/P69': 'educated at',
    'http://www.wikidata.org/prop/direct/P19': 'place of birth',
    'http://www.wikidata.org/prop/direct/P108': 'employer',
    'http://www.wikidata.org/prop/direct/P527': 'has part',
}
for uri, cnt in sorted(pred_counts.items(), key=lambda x: -x[1])[:10]:
    label = PRED_LABELS.get(uri, uri.split('/')[-1])
    print(f'  {label:30s}: {cnt}')
